In [2]:
pip install yfinance --upgrade

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import yfinance as yf
import datetime as dt
from datetime import date
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA
from deap import base, creator, tools, algorithms
import random


In [3]:
START = "2011-01-01"
TODAY = "2021-01-01"
def load_data(ticker):
    data = yf.download(ticker, START, TODAY)
    data.reset_index(inplace=True)
    return data

In [4]:
def process_data(df):

    # Simple Moving Averages
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['SMA_200'] = df['Close'].rolling(window=200).mean()

    # Exponential and Weighted Moving Averages
    df['EMA'] = df['Close'].ewm(span=14, adjust=False).mean()
    df['DEMA'] = 2 * df['Close'].ewm(span=14).mean() - df['Close'].ewm(span=28).mean()
    df['WMA'] = df['Close'].rolling(window=14).apply(lambda x: np.dot(x, np.arange(1, 15)) / np.sum(np.arange(1, 15)))
    
    # Triple EMA (TEMA)
    ema1 = df['Close'].ewm(span=14, adjust=False).mean()
    ema2 = ema1.ewm(span=14, adjust=False).mean()
    ema3 = ema2.ewm(span=14, adjust=False).mean()
    df['TEMA'] = 3 * ema1 - 3 * ema2 + ema3

    # TRIMA
    df['TRIMA'] = df['Close'].rolling(window=14).mean()

    # RSI
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # CMO
    df['CMO'] = 100 * (df['Close'].diff(14).rolling(14).mean() / df['Close'].diff(14).abs().rolling(14).sum())

    # ATR
    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift()).abs()
    low_close = (df['Low'] - df['Close'].shift()).abs()
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['ATR'] = true_range.rolling(window=14).mean()

    # ROC
    df['ROC'] = df['Close'].pct_change(periods=12) * 100
    df['ROCR'] = df['Close'] / df['Close'].shift(12)
    df['ROCR100'] = df['ROCR'] * 100

    # MACD
    ema_fast = df['Close'].ewm(span=12, adjust=False).mean()
    ema_slow = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_fast - ema_slow
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_signal']

    # Momentum
    df['Momentum'] = df['Close'] - df['Close'].shift(10)

    # TRIX
    trix_ema1 = df['Close'].ewm(span=15).mean()
    trix_ema2 = trix_ema1.ewm(span=15).mean()
    trix_ema3 = trix_ema2.ewm(span=15).mean()
    df['TRIX'] = trix_ema3.pct_change() * 100

    # Aroon
    df['AroonUp'] = df['High'].rolling(14).apply(lambda x: 100 * (x.argmax() / 13), raw=True)
    df['AroonDown'] = df['Low'].rolling(14).apply(lambda x: 100 * (x.argmin() / 13), raw=True)
    df['AroonOscillator'] = df['AroonUp'] - df['AroonDown']

    # CCI
    tp = (df['High'] + df['Low'] + df['Close']) / 3
    sma_tp = tp.rolling(20).mean()
    mad = (tp - sma_tp).abs().rolling(20).mean()
    df['CCI'] = (tp - sma_tp) / (0.015 * mad)

    # Typical, Median, Weighted Close
    df['TYPPRICE'] = tp
    df['WCLPRICE'] = (df['High'] + df['Low'] + 2 * df['Close']) / 4
    df['MEDPRICE'] = (df['High'] + df['Low']) / 2

    # OBV
    df['OBV'] = np.where(df['Close'] > df['Close'].shift(), df['Volume'],
                         -df['Volume']).cumsum()

    # MFI
    mfi_tp = (df['High'] + df['Low'] + df['Close']) / 3
    mf = mfi_tp * df['Volume']
    df['MFI'] = mf.rolling(14).sum() / df['Volume'].rolling(14).sum()

    # Donchian Mid
    df['DonchianMid'] = (df['High'].rolling(20).max() + df['Low'].rolling(20).min()) / 2

    # CLV
    df['CLV'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])

    # DPO
    df['DPO_V'] = df['Close'].rolling(20).mean() - df['Close']

    # KST
    roc1 = df['Close'].pct_change(10) * 100
    roc2 = df['Close'].pct_change(15) * 100
    roc3 = df['Close'].pct_change(20) * 100
    roc4 = df['Close'].pct_change(30) * 100
    df['KST'] = roc1 + roc2 + roc3 + roc4

    # Chaikin A/D
    df['AD'] = (2 * df['Close'] - df['High'] - df['Low']) / (df['High'] - df['Low']) * df['Volume']
    df['ChaikinAD'] = df['AD'].cumsum()

    # Chaikin Volatility
    df['ChaikinVolatility'] = df['High'].rolling(10).max() - df['Low'].rolling(10).min()

    # VHF
    df['VHF'] = (df['High'].rolling(28).max() - df['Low'].rolling(28).min()) / df['Close'].rolling(28).mean()

    # EVWMA
    df['EVWMA'] = (df['Close'] * df['Volume']).ewm(span=14).mean()

    # ZLEMA
    df['ZLEMA'] = df['Close'].ewm(span=14).mean() - df['Close'].shift(14)

    # WAD
    df['WAD'] = ((2 * df['Close'] - df['High'] - df['Low']) / (df['High'] - df['Low'])) * df['Volume']
    df['WAD'] = df['WAD'].cumsum()

    # HMA
    df['HMA'] = df['Close'].rolling(14).mean()

    # SMI
    df['SMI'] = 100 * (df['Close'].ewm(span=8).mean() - df['Close'].ewm(span=17).mean()) / df['Close'].rolling(18).std()

    # BOP
    df['BOP'] = (df['Close'] - df['Open']) / (df['High'] - df['Low'])

    # Williams %R
    high14 = df['High'].rolling(14).max()
    low14 = df['Low'].rolling(14).min()
    df['WILLR'] = -100 * (high14 - df['Close']) / (high14 - low14)

    # Stochastic %K and %D
    low_min = df['Low'].rolling(window=14).min()
    high_max = df['High'].rolling(window=14).max()
    df['STOCHK'] = 100 * ((df['Close'] - low_min) / (high_max - low_min))
    df['STOCHD'] = df['STOCHK'].rolling(window=3).mean()

    return df


In [5]:
def svc_with_ga(data, kernel='rbf', generations=20, population_size=30):
    df = data.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    # Feature Engineering
    df = process_data(df)

    # Target variable (ternary classification)
    df['pct_change'] = df['Close'].pct_change()
    
    quartiles = np.percentile(df['pct_change'].iloc[257:2272].dropna(), [25, 50, 75])
    
    y = np.where(
        df['pct_change'] < quartiles[0], 0,
        np.where(df['pct_change'] < 0, 1,
        np.where(df['pct_change'] < quartiles[2], 2, 3))
    )
    y = np.delete(y, 0)
    

    # Define feature set
    feature_cols = [
    'SMA_50', 'SMA_200', 'EMA', 'DEMA', 'WMA', 'TEMA', 'TRIMA',
    'RSI', 'CMO', 'ATR', 'ROC', 'ROCR', 'ROCR100', 'MACD', 'MACD_signal', 'MACD_hist',
    'Momentum', 'TRIX',
    'AroonUp', 'AroonDown', 'AroonOscillator',
    'CCI', 'TYPPRICE', 'WCLPRICE', 'MEDPRICE',
    'OBV', 'MFI',
    'DonchianMid', 'CLV', 'DPO_V', 'KST',
    'ChaikinAD', 'ChaikinVolatility',
    'VHF', 'EVWMA', 'ZLEMA',
    'WAD', 'HMA', 'SMI',
    'BOP', 'WILLR', 'STOCHK', 'STOCHD']


    X = df[feature_cols]

    # Remove rows with NaN values from rolling calculations
    X = X[257:2272]
    y = y[257:2272]

    # Train/test split (chronologically)
    train_size = int(len(X) * 0.8)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- GA for Feature Selection ---
    n_features = X_train.shape[1]

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)
    toolbox = base.Toolbox()
    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_features)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    def evaluate(individual):
        selected = [i for i, bit in enumerate(individual) if bit == 1]
        if len(selected) == 0:
            return 0.0,

        X_train_sub = X_train_scaled[:, selected]
        X_test_sub = X_test_scaled[:, selected]

        clf = SVC(kernel=kernel, decision_function_shape='ovo')
        clf.fit(X_train_sub, y_train)
        y_pred = clf.predict(X_test_sub)
        acc = accuracy_score(y_test, y_pred)
        return acc,

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
    toolbox.register("select", tools.selTournament, tournsize=3)

    def run_ga():
        pop = toolbox.population(n=population_size)
        hof = tools.HallOfFame(1)
        stats = tools.Statistics(lambda ind: ind.fitness.values)
        stats.register("avg", np.mean)
        stats.register("max", np.max)

        pop, logbook = algorithms.eaSimple(
            pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=generations,
            stats=stats, halloffame=hof, verbose=False
        )
        return hof[0], logbook

    best_individual, log = run_ga()
    selected_indices = [i for i, bit in enumerate(best_individual) if bit == 1]

    print("🔎 Selected Features:")
    print([feature_cols[i] for i in selected_indices])

    # Final model with selected features
    final_model = SVC(kernel=kernel, decision_function_shape='ovo')
    final_model.fit(X_train_scaled[:, selected_indices], y_train)
    y_pred = final_model.predict(X_test_scaled[:, selected_indices])

    return quartiles,y_test, y_pred, [feature_cols[i] for i in selected_indices]

In [6]:
data=load_data('AAPL')
quartiles,svm_test, svm_pred, selected_features = svc_with_ga(data,kernel='rbf', generations=20, population_size=30)
print(quartiles)
print(accuracy_score(svm_test,svm_pred))
print(svm_pred)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


🔎 Selected Features:
['SMA_50', 'EMA', 'ATR', 'ROC', 'ROCR', 'ROCR100', 'MACD_hist', 'Momentum', 'AroonDown', 'AroonOscillator', 'WCLPRICE', 'OBV', 'DonchianMid', 'DPO_V', 'ChaikinVolatility', 'EVWMA', 'ZLEMA', 'HMA', 'SMI', 'BOP', 'WILLR', 'STOCHK', 'STOCHD']
[-0.00659215  0.00088769  0.00963058]
0.3424317617866005
[2 1 1 2 1 1 1 1 0 0 0 0 0 0 0 1 2 1 2 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 3 2 2 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 1 1 1 2 0 1 0 3 1 3 1 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 0 0 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 2 2 2 3 2 2 2 1 1 1 1 1 1 1 2 2 2 2 2 2 1 2 2 2 2 2 2
 2 1 1 1 2 2 2 2 2 2 2 2 1 1 1 2 2 2 3 3 3 3 2 3 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 1 1 3 3 2 3 1 0 0 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 2 2 2 1 1 1 1
 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 2 2 2 3 1 1 2 2 1 1 1 1 1 1 1 0
 3 3 3 

In [7]:
correct = 0
total = len(svm_test)

# Iterate through each item in the arrays
for i in range(total):
    true_value = svm_test[i]
    pred_value = svm_pred[i]
    
    # Check if the true value is 0 or 1
    if true_value in [0, 1]:
        if pred_value in [0, 1]:
            correct += 1
    
    # Check if the true value is 2 or 3
    elif true_value in [2, 3]:
        if pred_value in [2, 3]:
            correct += 1

# Calculate accuracy
accuracy = correct / total
print("Custom Accuracy:", accuracy)

Custom Accuracy: 0.56575682382134


In [8]:
temp_test=svm_test
temp_pred_svc=svm_pred

accuracy_per_class = []

# Iterate over each unique class (0, 1, 2, 3)
for class_label in np.unique(temp_test):
    # Get the indices where the true label is the current class
    indices = np.where(temp_test == class_label)[0]
    
    # Extract the corresponding true and predicted labels
    true_labels = temp_test[indices]
    pred_labels = temp_pred_svc[indices]
    
    # Calculate the accuracy for the current class
    accuracy = accuracy_score(true_labels, pred_labels)
    
    # Store the result
    accuracy_per_class.append(accuracy)

# Convert the list to a numpy array or just print the results
accuracy_per_class = np.array(accuracy_per_class)
print("Accuracy per class:", accuracy_per_class)

Accuracy per class: [0.04901961 0.26666667 0.4122807  0.58928571]


In [9]:
from sklearn.metrics import classification_report
print(classification_report(svm_test, svm_pred, digits=4))


              precision    recall  f1-score   support

           0     0.3571    0.0490    0.0862       102
           1     0.2439    0.2667    0.2548        75
           2     0.4052    0.4123    0.4087       114
           3     0.3455    0.5893    0.4356       112

    accuracy                         0.3424       403
   macro avg     0.3379    0.3293    0.2963       403
weighted avg     0.3464    0.3424    0.3059       403



In [10]:
soft_accuracy_per_class = []

class_labels = np.unique(temp_test)

for class_label in class_labels:
    indices = np.where(temp_test == class_label)[0]
    true_labels = temp_test[indices]
    pred_labels = temp_pred_svc[indices]

    # Define acceptable predictions based on clarified logic
    if class_label in [0, 1]:
        acceptable = [0, 1]
    elif class_label in [2, 3]:
        acceptable = [2, 3]

    soft_correct = np.sum(np.isin(pred_labels, acceptable))
    soft_total = len(true_labels)
    soft_accuracy = soft_correct / soft_total
    soft_accuracy_per_class.append(soft_accuracy)

soft_accuracy_per_class = np.array(soft_accuracy_per_class)
print("🟢 Soft Accuracy per class:", soft_accuracy_per_class)

🟢 Soft Accuracy per class: [0.25490196 0.30666667 0.78947368 0.79464286]


LSTM

In [ ]:
from deap import base, creator, tools, algorithms
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, confusion_matrix
from tensorflow.keras import backend as K
import numpy as np
import random

def lstm_with_ga(data, pop_size=8, n_gen=8):
    # Step 1: Preprocessing
    df = data.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df = process_data(df)
    df['pct_change'] = df['Close'].pct_change()

    # four-class label 
    quartiles = np.percentile(df['pct_change'].iloc[200:1700].dropna(), [25,50,75])
    y = np.where(
        df['pct_change'] < quartiles[0], 0,
        np.where(df['pct_change'] < quartiles[1], 1,
        np.where(df['pct_change'] < quartiles[2], 2, 3))
    )
    y = np.delete(y, 0)

    feature_cols = [
        'SMA_50','SMA_200','EMA','DEMA','WMA','TEMA','TRIMA',
        'RSI','CMO','ATR','ROC','ROCR','ROCR100',
        'MACD','MACD_signal','MACD_hist',
        'Momentum','TRIX','AroonUp','AroonDown','AroonOscillator',
        'CCI','TYPPRICE','WCLPRICE','MEDPRICE',
        'OBV','MFI','DonchianMid','CLV','DPO_V','KST',
        'ChaikinAD','ChaikinVolatility',
        'VHF','EVWMA','ZLEMA',
        'WAD','HMA','SMI','BOP','WILLR','STOCHK','STOCHD'
    ]
    df = df.iloc[257:2272][feature_cols].copy()
    y = y[257:2272]

    # Step 2: GA setup
    n_feats = len(feature_cols)
    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()
    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_feats)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    def evaluate(individual):
        sel = [i for i, bit in enumerate(individual) if bit]
        if not sel:
            return 0.0,

        # build X-train/test for these sel features
        X = df.iloc[:, sel].values
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        # reshape to (samples, timesteps, 1)
        Xtr = Xtr.reshape((Xtr.shape[0], Xtr.shape[1], 1))
        Xte = Xte.reshape((Xte.shape[0], Xte.shape[1], 1))
        timesteps = Xtr.shape[1]

        # Simple 4-class LSTM
        model = Sequential()
        model.add(LSTM(64, return_sequences=True, input_shape=(timesteps,1)))
        model.add(LSTM(32))
        model.add(Dense(16, activation='relu'))
        model.add(Dropout(0.3))
        model.add(Dense(4, activation='softmax'))
        model.compile(optimizer=Adam(0.001),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
        model.fit(Xtr, ytr, epochs=5, batch_size=64, verbose=0)
        _, acc = model.evaluate(Xte, yte, verbose=0)
        K.clear_session()
        return acc,

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)

    #  run GA
    pop = toolbox.population(pop_size)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("max", np.max)

    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2,
                        ngen=n_gen, stats=stats, halloffame=hof, verbose=True)

    best = hof[0]
    sel_idx = [i for i, b in enumerate(best) if b]
    best_feats = [feature_cols[i] for i in sel_idx]
    print("Selected features:", best_feats)

    # Step 4: final model on best_feats
    X = df[best_feats].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr).reshape((-1, len(best_feats), 1))
    Xte = scaler.transform(Xte).reshape((-1, len(best_feats), 1))

    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=(len(best_feats),1)))
    model.add(LSTM(32))
    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(4, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=Adam(0.001), metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(Xtr, ytr, validation_data=(Xte,yte),
              epochs=15, batch_size=100, callbacks=[es], verbose=1)

    y_prob = model.predict(Xte)
    y_pred = np.argmax(y_prob, axis=1)
    acc_final = accuracy_score(yte, y_pred)
    cm = confusion_matrix(yte, y_pred)

    return yte, y_pred, acc_final, cm, best_feats




In [12]:
y_test, y_pred, acc, cm, best_features = lstm_with_ga(load_data("AAPL"))
print("Confusion Matrix:\n", cm)
print("Test Accuracy:", acc)


[*********************100%***********************]  1 of 1 completed
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppD

gen	nevals	avg     	max     
0  	8     	0.255273	0.280397


c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppD

1  	8     	0.266129	0.28536 


...

7  	4     	0.278536	0.307692


c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\Linyi Zhang\AppD

8  	4     	0.288151	0.307692
✅ Selected features: ['SMA_50', 'EMA', 'WMA', 'TEMA', 'ROC', 'ROCR', 'ROCR100', 'MACD', 'MACD_signal', 'AroonUp', 'AroonDown', 'OBV', 'DPO_V', 'ChaikinVolatility', 'VHF', 'EVWMA', 'WAD', 'BOP']
Epoch 1/15


c:\Users\Linyi Zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.2726 - loss: 1.3837 - val_accuracy: 0.2432 - val_loss: 1.3818
Epoch 2/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2853 - loss: 1.3797 - val_accuracy: 0.2531 - val_loss: 1.3825
Epoch 3/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2923 - loss: 1.3731 - val_accuracy: 0.2804 - val_loss: 1.3812
Epoch 4/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.2754 - loss: 1.3713 - val_accuracy: 0.2730 - val_loss: 1.3790
Epoch 5/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3000 - loss: 1.3704 - val_accuracy: 0.2878 - val_loss: 1.3806
Epoch 6/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.2986 - loss: 1.3681 - val_accuracy: 0.3176 - val_loss: 1.3776
Epoch 7/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.2753 - loss: 1.3736 - val_accuracy: 0.3176 - val_loss: 1.3769
Epoch 8/15
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.2948 - loss: 1.3700 - val_accuracy: 0.2978 - val_loss: 1.

In [13]:
print(accuracy_score(y_test,y_pred))

0.31017369727047145


In [20]:
print(classification_report(y_test, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.3253    0.2523    0.2842       107
           1     0.3310    0.4747    0.3900        99
           2     0.2899    0.1905    0.2299       105
           3     0.2844    0.3370    0.3085        92

    accuracy                         0.3102       403
   macro avg     0.3076    0.3136    0.3031       403
weighted avg     0.3081    0.3102    0.3016       403



In [18]:
correct = 0
total = len(y_test)

# Iterate through each item in the arrays
for i in range(total):
    true_value = y_test[i]
    pred_value = y_pred[i]
    
    # Check if the true value is 0 or 1
    if true_value in [0, 1]:
        if pred_value in [0, 1]:
            correct += 1
    
    # Check if the true value is 2 or 3
    elif true_value in [2, 3]:
        if pred_value in [2, 3]:
            correct += 1

# Calculate accuracy
accuracy = correct / total
print("Custom Accuracy:", accuracy)

Custom Accuracy: 0.5508684863523573


In [ ]:
temp_test=y_test
temp_pred_svc=y_pred

accuracy_per_class = []

# Iterate over each unique class (0, 1, 2, 3)
for class_label in np.unique(temp_test):
    # Get the indices where the true label is the current class
    indices = np.where(temp_test == class_label)[0]
    
    # Extract the corresponding true and predicted labels
    true_labels = temp_test[indices]
    pred_labels = temp_pred_svc[indices]
    
    # Calculate the accuracy for the current class
    accuracy = accuracy_score(true_labels, pred_labels)
    
    # Store the result
    accuracy_per_class.append(accuracy)

# Convert the list to a numpy array or just print the results
accuracy_per_class = np.array(accuracy_per_class)
print("Accuracy per class:", accuracy_per_class)

Accuracy per class: [0.25233645 0.47474747 0.19047619 0.33695652]


In [16]:
soft_accuracy_per_class = []

class_labels = np.unique(temp_test)

for class_label in class_labels:
    indices = np.where(temp_test == class_label)[0]
    true_labels = temp_test[indices]
    pred_labels = temp_pred_svc[indices]

    # Define acceptable predictions based on clarified logic
    if class_label in [0, 1]:
        acceptable = [0, 1]
    elif class_label in [2, 3]:
        acceptable = [2, 3]

    soft_correct = np.sum(np.isin(pred_labels, acceptable))
    soft_total = len(true_labels)
    soft_accuracy = soft_correct / soft_total
    soft_accuracy_per_class.append(soft_accuracy)

soft_accuracy_per_class = np.array(soft_accuracy_per_class)
print("🟢 Soft Accuracy per class:", soft_accuracy_per_class)

🟢 Soft Accuracy per class: [0.54205607 0.67676768 0.51428571 0.4673913 ]


In [17]:
print(y_pred)

[1 1 1 1 2 0 3 3 1 1 0 1 1 1 0 2 2 3 1 2 0 3 0 2 3 1 1 3 2 2 3 3 3 2 1 1 1
 0 3 1 3 0 0 2 2 2 3 0 3 0 0 1 3 2 1 1 0 0 1 2 0 2 1 3 2 1 3 0 3 2 3 1 1 3
 1 1 1 1 1 3 2 3 1 2 1 0 3 2 2 1 1 3 1 0 3 3 2 0 3 3 3 1 3 1 3 3 2 3 1 3 2
 1 0 2 1 1 0 0 1 1 0 3 0 1 1 2 3 0 3 3 1 0 0 3 1 0 1 1 0 1 1 2 3 1 1 2 1 3
 1 3 0 2 1 1 2 3 1 2 3 1 3 0 2 3 3 3 0 2 3 0 1 0 1 0 1 1 0 0 3 3 0 2 2 1 0
 3 1 3 1 1 2 1 1 3 0 2 3 1 2 1 3 2 0 0 1 1 3 2 3 1 0 3 0 0 0 1 1 1 3 1 0 0
 0 1 3 1 2 3 3 0 0 2 2 1 3 2 0 3 3 3 0 3 1 0 1 1 3 2 1 2 1 3 1 1 1 0 3 1 1
 1 2 1 0 1 0 1 3 0 0 2 1 3 0 1 2 1 0 1 3 3 3 1 2 0 3 3 3 2 1 0 0 2 0 3 1 1
 1 1 3 0 0 1 1 2 3 0 3 1 1 1 1 2 3 3 1 0 2 2 1 2 2 0 1 1 3 0 2 0 2 3 3 1 1
 0 1 3 1 3 3 1 1 3 0 1 2 3 1 1 0 0 3 1 3 2 3 3 1 2 1 3 3 0 3 0 1 1 1 1 0 1
 1 0 3 2 0 1 0 0 0 3 2 2 1 1 1 2 1 3 1 3 1 1 3 3 3 2 3 3 1 2 2 1 3]


XGB

In [ ]:
import numpy as np
import pandas as pd
import random
from deap import base, creator, tools, algorithms
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

def xgb_multiclass_ga(data, pop_size=20, n_gen=10, seed=42):
    # --- 1. Preprocessing 
    df = data.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df = process_data(df)  # you must define this to add your indicators
    df['pct_change'] = df['Close'].pct_change()
    
    # Quartile 
    pct = df['pct_change'].iloc[200:1700].dropna().values
    q1, q2, q3 = np.percentile(pct, [25, 50, 75])
    df['y_cat'] = np.select(
        [
            df['pct_change'] < q1,
            (df['pct_change'] >= q1) & (df['pct_change'] < q2),
            (df['pct_change'] >= q2) & (df['pct_change'] < q3)
        ],
        [0, 1, 2],
        default=3
    )
    y = df['y_cat'].values[1:]  # drop first NaN
    
    #  train/test split ---
    feature_cols = [
        'High','Open','Low','Close',
        'SMA_50','SMA_200','EMA','DEMA','WMA','TEMA','TRIMA',
        'RSI','CMO','ATR','ROC','ROCR','ROCR100',
        'MACD','MACD_signal','MACD_hist',
        'Momentum','TRIX','AroonUp','AroonDown','AroonOscillator',
        'CCI','TYPPRICE','WCLPRICE','MEDPRICE',
        'OBV','MFI','DonchianMid','CLV','DPO_V','KST',
        'ChaikinAD','ChaikinVolatility',
        'VHF','EVWMA','ZLEMA',
        'WAD','HMA','SMI','BOP','WILLR','STOCHK','STOCHD'
    ]
    df = df.iloc[257:2272].copy()
    y = y[257:2272]
    df = df[[c for c in feature_cols if c in df.columns]]
    actual_features = list(df.columns)
    
    X = df.values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, shuffle=False
    )
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)
    
    # GA 
    n_feats = X_train.shape[1]
    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)
    
    toolbox = base.Toolbox()
    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register("individual", tools.initRepeat, creator.Individual,
                     toolbox.attr_bool, n=n_feats)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    def evaluate(individual):
        # select features
        sel = [i for i, bit in enumerate(individual) if bit]
        if not sel:
            return 0.0,
        Xtr = X_train[:, sel]
        Xte = X_test[:, sel]
        # multiclass XGBoost
        model = XGBClassifier(
            objective='multi:softprob',
            num_class=4,
            eval_metric='mlogloss',
            use_label_encoder=False,
            random_state=seed,
            verbosity=0
        )
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        return accuracy_score(y_test, pred),
    
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    # GA
    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(1)
    algorithms.eaSimple(
        pop, toolbox,
        cxpb=0.5, mutpb=0.2,
        ngen=n_gen,
        halloffame=hof,
        verbose=True
    )
    
    best = hof[0]
    sel_idx = [i for i, b in enumerate(best) if b]
    best_features = [actual_features[i] for i in sel_idx]
    print("Selected features:", best_features)
    
    # inal model training 
    final_model = XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=seed,
        verbosity=0
    )
    final_model.fit(X_train[:, sel_idx], y_train)
    y_pred = final_model.predict(X_test[:, sel_idx])
    
    return y_test, y_pred, best_features

 



In [7]:
data=load_data('AAPL')
y_test, y_pred, features = xgb_multiclass_ga(data, pop_size=30, n_gen=20)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

C:\Users\Linyi Zhang\AppData\Local\Temp\ipykernel_17868\946618935.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, START, TODAY)
[*********************100%***********************]  1 of 1 completed


gen	nevals
0  	30    
1  	16    
2  	22    
3  	17    
4  	21    
5  	21    
6  	20    
7  	18    
8  	19    
9  	19    
10 	18    
11 	19    
12 	19    
13 	15    
14 	12    
15 	17    
16 	13    
17 	19    
18 	14    
19 	12    
20 	14    
✅ Selected features: [('High', 'AAPL'), ('EMA', ''), ('DEMA', ''), ('WMA', ''), ('TEMA', ''), ('RSI', ''), ('CMO', ''), ('ROCR', ''), ('MACD', ''), ('MACD_signal', ''), ('Momentum', ''), ('AroonOscillator', ''), ('CCI', ''), ('TYPPRICE', ''), ('MFI', ''), ('DPO_V', ''), ('KST', ''), ('ChaikinVolatility', ''), ('EVWMA', ''), ('ZLEMA', ''), ('BOP', ''), ('WILLR', ''), ('STOCHK', ''), ('STOCHD', '')]
Accuracy: 0.3027295285359802
Confusion Matrix:
 [[18 29 24 31]
 [10 33 18 22]
 [10 36 27 26]
 [15 37 23 44]]


In [8]:
print(classification_report(y_test, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.3396    0.1765    0.2323       102
           1     0.2444    0.3976    0.3028        83
           2     0.2935    0.2727    0.2827        99
           3     0.3577    0.3697    0.3636       119

    accuracy                         0.3027       403
   macro avg     0.3088    0.3041    0.2953       403
weighted avg     0.3140    0.3027    0.2980       403



In [9]:
correct = 0
total = len(y_test)

# Iterate through each item in the arrays
for i in range(total):
    true_value = y_test[i]
    pred_value = y_pred[i]
    
    # Check if the true value is 0 or 1
    if true_value in [0, 1]:
        if pred_value in [0, 1]:
            correct += 1
    
    # Check if the true value is 2 or 3
    elif true_value in [2, 3]:
        if pred_value in [2, 3]:
            correct += 1

# Calculate accuracy
accuracy = correct / total
print("Custom Accuracy:", accuracy)

Custom Accuracy: 0.5210918114143921
